# Getting Started with CHI@Edge

**What this notebook covers:**
1. Getting started:
   * Launch a first container
   * Interacting with the container
   * Assigning a public IP
   * Uploading files

**Note: this tutorial makes use of [python-chi](https://python-chi.readthedocs.io/en/latest/), Chameleon's Python API!**

In [1]:
import chi
# Before we go any further, we need to select which Chameleon site we will be using.
chi.use_site("CHI@Edge")

Now using CHI@Edge:
URL: https://chi.edge.chameleoncloud.org
Location: University of Chicago, Chicago, Illinois, USA
Support contact: help@chameleoncloud.org


Next, select which Chameleon project/allocation you will use. **Use the CH-XXXXXX** identitifer, **not** the project nickname.

You can find your project ID on the user dashboard here: https://chameleoncloud.org/user/dashboard/

In [ ]:
chi.set("project_name", "CHI-241397") # Use your respective project ID here

From here on out, we will primarily be using the `container` module in python-chi to interact with the Chameleon Container API.

In [3]:
from chi import container
from chi import lease

## Getting started

Some helpful tips when editing and working in Jupyter code cells:

* Tab-completion is available! Try pressing `<tab>` when typing to get an autocomplete list.
* When in a function invocation, press `<shift>+<tab>` to get documentation for the function, e.g., supported arguments.

### Reserving edge devices

On CHI@Edge, you reserve edge devices for your exclusive use, enabling you to more strictly control performance conditions and take advantage of the entire device. This is done via Chameleon's leasing system. If you've used the Blazar lease system in the past on a separate Chameleon site, it works pretty much the same way as you're used to. Here, we will reserve two Raspberry Pi model 4's.

**Note**: To see what edge hardware is currently hosted in CHI@Edge, check out the [edge hardware discovery table](https://www.chameleoncloud.org/edge/hardware/). After examining the types of devices we have, you can then see which of them are available on the resource [calendar](https://chi.edge.chameleoncloud.org/project/leases/calendar/device/) available within the CHI@Edge GUI. To reserve a specific device from within the GUI directly, use the UUID displayed on the hardware discovery table to select a device by adding the uid resource property when reserving devices on the create lease form. Otherwise, follow the instructions below.

In [4]:
import os

# get your username, just used to name leases something identifiable
username = os.environ.get("USER")

# machine name refers to the "type" of device
machine_name = "raspberrypi4-64"

# if you want to reserve a specific device, device name refers to the name of the actual device you want to reserve
device_name = "check the resource calendar for an available device and copy over its name"

# get dates for lease start and end
start, end = lease.lease_duration(days=1)

# make a unique name for the lease
lease_name = f"{username}-{machine_name}-{start}"

reservations = []
lease.add_device_reservation(reservations, count=2, machine_name=machine_name)
# (To reserve a specific device) lease.add_device_reservation(reservations, count=1, machine_name=machine_name, device_name=device_name)
container_lease = lease.create_lease(lease_name, reservations)
lease_id = container_lease["id"]

print(f"created lease with name {lease_name} and uuid {lease_id}, waiting for it to start. This can take up to 60s.")
lease.wait_for_active(lease_id)
print("Done!")

created lease with name michaelly_cpp_edu-raspberrypi4-64-2025-05-01 18:45 and uuid e1eb8367-c057-46e4-af4f-dad3ad239ce8, waiting for it to start. This can take up to 60s.
Done!


### Launching a first container

For an introduction, we'll launch a pretty simple container, which will just start a webserver that serves a directory of files. Python's `http.server` module is great for this. We will use the `python:3.9-slim` Docker image, which is a version of the "slim" (less binaries/libraries included) official Python Docker image. This image includes a "manifest" file, so that multiple architectures can be provided under one tag. It is important to understand which architecture your device uses; Raspberry Pis (which we use in this webinar) use ARM64, other devices may use other architectures and therefore will need different Docker images.

If you are a bit experienced with Docker, this should look pretty familiar:

In [ ]:
print("Requesting container ... This may take a while as the container image is being downloaded")

# Set a name for the container. Because CHI@Edge uses Kubernetes, ensure that underscores aren't in the name
container_name = f"tutorial-{machine_name}-webserver".replace("_","-")

try:
    my_container = container.create_container(
        container_name, 
        image="python:3.9-slim",
        command=["python", "-m", "http.server", "8000"],
        workdir="/var/www/html",
        exposed_ports=[8000, 8080, 6000, 5000, 6001, 6002, 6003, 5001, 5002, 5003], # Extra ports for the webserver DO NOT DELETE
        reservation_id=lease.get_device_reservation(lease_id),
        platform_version=2,
    )
except RuntimeError as ex:
    print(ex)
    print(f"please stop and/or delete {container_name} and try again")
else:
    print(f"Successfully created container: {container_name}!")

Requesting container ... This may take a while as the container image is being downloaded
Waiting for container f4a9d108-c349-4a1c-83ac-173cd2bafe8f status to turn to Running. This can take a while depending on the image
Successfully created container: tutorial-raspberrypi4-64-webserver!


### Interacting with the container

It is possible to run one-off commands with `execute()`. This is not well-suited for long-running commands, but can be good for quick sanity checks.

In [24]:
cmd = "ls -l /var/www/html"
print(cmd)

print(container.execute(my_container.uuid, cmd)["output"])

ls -l /var/www/html
total 0



### Assigning a public IP

When you assign a public IP address, any exposed ports on your container can be reached over the public internet.

*Note*: sometimes it takes a while (a minute or so) for the floating IP to become "live".
If it takes longer than this, try deleting your container, then recreating it and associating the IP again. Second and further tries should be quicker.

In [25]:
public_ip = container.associate_floating_ip(my_container.uuid)

print(f"http://{public_ip}:8000")

http://129.114.34.183:8000


### Uploading a file or directory to your container

You can upload a file or directory of files directly to the running container with `upload()`. Remember that these devices are very memory-constrained; you should not be uploading directories that are large! Keeping uploads below 1MB is a good general rule; this is mostly designed for uploading configuration or support files.

In [ ]:
# Uploading a file
container.upload(my_container.uuid, "./edge_script.py", "/var/www/html")
container.upload(my_container.uuid, "./OFFICIAL_DEMO_MODEL.tflite", "/var/www/html")

print("Files uploaded!")
print(container.execute(my_container.uuid, "ls -l /var/www/html")["output"])
print(f"use this link to access your uploaded page http://{public_ip}:8000")

Files uploaded!
total 3216
-rw-rw-r-- 1 1000 users 3272916 May  2 00:52 OFFICIAL_DEMO_MODEL.tflite
-rw-r--r-- 1 1000 users   13124 May  2 00:52 edge_script.py

use this link to access your uploaded page http://129.114.34.183:8000
